<img src="logos/Logo_de_la_Facultad_Experimental_de_Ciencia_y_Tecnología.svg.png"
     width="250"
     style="display: block; margin-left: auto; margin-right: auto;">

# Titanic – Modeling and Model Selection

## Objective

This notebook develops the first modeling stage of the Titanic project.

The goals are:

1. Establish baseline performance.
2. Compare multiple classification models.
3. Select the most promising model using cross-validation.
4. Improve the selected model through hyperparameter tuning.
5. Evaluate the final model on the test set.
6. Interpret results and identify possible failure modes.

---

## Methodological Principles

This notebook follows a professional machine learning workflow:

- The preprocessing pipeline is reused from Notebook 2.
- The test set is kept untouched until the final evaluation.
- Baseline models are established before tuning.
- Model selection is based on quantitative metrics.
- Final conclusions must justify why a model is chosen.

---

## Models to Evaluate

### Baseline models
- Dummy Classifier
- Logistic Regression
- Decision Tree

### Candidate models
- Random Forest
- HistGradientBoosting

---

## Main Evaluation Metrics

The main metric for model selection will be **F1-score**.

Additional metrics:
- Accuracy
- ROC-AUC (AUC = Area Under the Curve)

This is important because accuracy alone may be misleading in imbalanced classification problems.

# General Structure of a Scikit Learn Model

```
from sklearn.modelo import Modelo

# 1. Instanciar (definir hiperparámetros)
model = Modelo(param1=..., param2=...)

# 2. Entrenar
model.fit(X_train, y_train)

# 3. Predecir
y_pred = model.predict(X_test)

# (Opcional)
probs = model.predict_proba(X_test)
```

# Imports and Configurations

In [35]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5

ARTIFACTS_DIR = Path("../artifacts")

CV = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

## Setup and Experiment Configuration

This cell imports the models, metrics, and utilities required for the modeling phase.

It also defines:

- a fixed random seed for reproducibility,
- the cross-validation strategy,
- and the evaluation metrics used throughout the notebook.

Cross-validation is configured with stratified folds in order to preserve
the class distribution across splits.

In [36]:
preprocess_pipeline = joblib.load(
    ARTIFACTS_DIR / "preprocess_pipeline.joblib"
)

X_train, X_test, y_train, y_test = joblib.load(
    ARTIFACTS_DIR / "split_data.joblib"
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target mean:", y_train.mean())
print("Test target mean:", y_test.mean())

Train shape: (712, 12)
Test shape: (179, 12)
Train target mean: 0.38342696629213485
Test target mean: 0.3854748603351955


## Loading Prepared Data and Preprocessing Pipeline

This cell loads the outputs generated in Notebook 1:

- the preprocessing pipeline,
- the training and test feature sets,
- and the target vectors.

This separation is intentional:
Notebook 1 is responsible for data preparation,
while Notebook 2 focuses strictly on modeling.

In [37]:
def make_model_pipeline(preprocess_pipeline, model):
    """
    Build a full modeling pipeline that combines preprocessing
    and classification in a single reproducible object.

    Parameters
    ----------
    preprocess_pipeline : sklearn transformer
        Preprocessing pipeline defined in Notebook 1.
    model : sklearn estimator
        Classification model.

    Returns
    -------
    sklearn.pipeline.Pipeline
        Full pipeline with preprocessing and model.
    """
    return Pipeline(
        steps=[
            ("preprocess", preprocess_pipeline),
            ("model", model),
        ]
    )

## Full Modeling Pipeline

This helper function creates a complete sklearn pipeline by combining:

1. the preprocessing steps defined in Notebook 2,
2. the classifier to be trained.

This is a critical design choice because it ensures that:

- preprocessing is applied consistently,
- transformations are learned only from training data,
- and data leakage is avoided during cross-validation and testing.

In [38]:
baseline_models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    ),
    "decision_tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
    ),
}

candidate_models = {
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
}

## Initial Model Set

This notebook evaluates two groups of models:

### Baseline models
These establish the minimum expected performance:
- Dummy Classifier
- Logistic Regression
- Decision Tree

### Candidate models
These are more powerful ensemble methods:
- Random Forest
- HistGradientBoosting

The goal is not to try every possible algorithm,
but to compare a small set of relevant models in a structured way.

In [39]:
def evaluate_model_cv(model_name, preprocess_pipeline, model, X_train, y_train):
    """
    Evaluate a classification model using stratified cross-validation.

    Parameters
    ----------
    model_name : str
        Name used to identify the model in the output table.
    model : sklearn estimator
        Classification model to evaluate.
    X_train : pandas.DataFrame
        Training feature matrix.
    y_train : pandas.Series
        Training target vector.

    Returns
    -------
    dict
        Dictionary containing the model name and mean cross-validation metrics.
    """
    model_pipeline = make_model_pipeline(preprocess_pipeline, model)

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
    )

    return {
        "model": model_name,
        "cv_accuracy_mean": np.mean(cv_results["test_accuracy"]),
        "cv_f1_mean": np.mean(cv_results["test_f1"]),
        "cv_roc_auc_mean": np.mean(cv_results["test_roc_auc"]),
    }

## Cross-Validation Evaluation Function

This function evaluates a model using stratified cross-validation.

For each model, it:

1. builds a full pipeline with preprocessing + classifier,
2. performs cross-validation on the training set,
3. computes the mean values of the selected evaluation metrics.

This design allows us to compare models under the same conditions
before making any tuning or final selection decisions.

In [40]:
baseline_results = []

for model_name, model in baseline_models.items():
    result = evaluate_model_cv(
        model_name=model_name,
        preprocess_pipeline=preprocess_pipeline,
        model=model,
        X_train=X_train,
        y_train=y_train,
    )
    baseline_results.append(result)

baseline_results_df = (
    pd.DataFrame(baseline_results)
    .sort_values(by="cv_f1_mean", ascending=False)
    .reset_index(drop=True)
)

baseline_results_df

,model,cv_accuracy_mean,cv_f1_mean,cv_roc_auc_mean
0,logistic_regression,0.823087,0.769401,0.874673
1,decision_tree,0.778046,0.709868,0.765877
2,dummy,0.616576,0.000000,0.500000


## Baseline Model Results

This table summarizes the performance of the baseline models.

The purpose of this stage is to establish a minimum reference point.
In particular, the Dummy Classifier tells us how well a trivial strategy performs,
while Logistic Regression and Decision Tree provide stronger but still simple baselines.

Any candidate model should clearly outperform these results.

In [41]:
candidate_results = []

for model_name, model in candidate_models.items():
    result = evaluate_model_cv(
        model_name=model_name,
        preprocess_pipeline=preprocess_pipeline,
        model=model,
        X_train=X_train,
        y_train=y_train,
    )
    candidate_results.append(result)

candidate_results_df = (
    pd.DataFrame(candidate_results)
    .sort_values(by="cv_f1_mean", ascending=False)
    .reset_index(drop=True)
)

candidate_results_df

,model,cv_accuracy_mean,cv_f1_mean,cv_roc_auc_mean
0,random_forest,0.813208,0.754871,0.871623
1,hist_gradient_boosting,0.814626,0.748608,0.888027


## Candidate Model Results

This table summarizes the performance of the stronger candidate models.

These models are expected to capture more complex relationships than the baselines.
However, they also introduce a greater risk of overfitting,
so their performance must be interpreted carefully.

In [10]:
all_results_df = (
    pd.concat([baseline_results_df, candidate_results_df], ignore_index=True)
    .sort_values(by="cv_f1_mean", ascending=False)
    .reset_index(drop=True)
)

all_results_df

,model,cv_accuracy_mean,cv_f1_mean,cv_roc_auc_mean
0,logistic_regression,0.823087,0.769401,0.874673
1,random_forest,0.813208,0.754871,0.871623
2,hist_gradient_boosting,0.814626,0.748608,0.888027
3,decision_tree,0.778046,0.709868,0.765877
4,dummy,0.616576,0.000000,0.500000


## Model Comparison Summary

This consolidated table compares all evaluated models using the same cross-validation setup.

The main criterion for model selection is **mean cross-validated F1-score**,
while Accuracy and ROC-AUC are used as complementary metrics.

The next step is to select the most promising model or models for hyperparameter tuning.

In [11]:
best_model_name = all_results_df.iloc[0]["model"]
best_model_name

'logistic_regression'

## Preliminary Best Model Selection

This cell identifies the model with the highest mean cross-validated F1-score.

This is only a preliminary selection.
The final decision will be made after hyperparameter tuning
and evaluation on the untouched test set.

# HYPERPARAMETER TUNING STEP

In [14]:
import optuna

## Hyperparameter Tuning with Optuna

At this stage, we optimize the strongest candidate models using Optuna.

The objective is to improve performance by searching for better
hyperparameter configurations while keeping the evaluation rigorous.

Important principles:

- Tuning is performed only on the training set.
- Cross-validation is used inside the optimization process.
- The test set remains untouched until the final evaluation.

In [15]:
def objective_random_forest(trial):
    """
    Optuna objective function for Random Forest.

    Parameters
    ----------
    trial : optuna.trial.Trial
        Current optimization trial.

    Returns
    -------
    float
        Mean cross-validated F1-score.
    """
    model = RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model_pipeline = make_model_pipeline(preprocess_pipeline, model)

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"f1": "f1"},
        n_jobs=-1,
        return_train_score=False,
    )

    return np.mean(cv_results["test_f1"])

## Random Forest Objective Function

This function defines the search space for Random Forest
and tells Optuna how to evaluate each trial.

For every sampled configuration:

1. a Random Forest model is created,
2. it is wrapped into the full preprocessing pipeline,
3. cross-validation is performed on the training data,
4. the mean F1-score is returned.

This ensures that tuning is based on the same metric used
for model comparison.

In [16]:
study_random_forest = optuna.create_study(direction="maximize")
study_random_forest.optimize(objective_random_forest, n_trials=30)

[I 2026-03-20 17:32:53,908] A new study created in memory with name: no-name-f8f6510e-1bfa-4d17-afbe-5053bbec5060
[I 2026-03-20 17:33:08,926] Trial 0 finished with value: 0.7590016271292276 and parameters: {'n_estimators': 102, 'max_depth': 9, 'min_samples_split': 17, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 0 with value: 0.7590016271292276.
[I 2026-03-20 17:33:25,117] Trial 1 finished with value: 0.7536711810106305 and parameters: {'n_estimators': 368, 'max_depth': 11, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 0.7590016271292276.
[I 2026-03-20 17:33:35,224] Trial 2 finished with value: 0.7724703489216846 and parameters: {'n_estimators': 279, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 0.7724703489216846.
[I 2026-03-20 17:33:36,009] Trial 3 finished with value: 0.7634195466428477 and parameters: {'n_estimators': 159, 'max_depth': 15, 'min

In [17]:
print("Best Random Forest F1:", study_random_forest.best_value)
print("Best Random Forest params:")
study_random_forest.best_params

Best Random Forest F1: 0.7743409887927803
Best Random Forest params:


{'n_estimators': 425,
 'max_depth': 9,
 'min_samples_split': 4,
 'min_samples_leaf': 2,
 'max_features': None}

## Random Forest Tuning Result

This output shows the best cross-validated F1-score found by Optuna
for Random Forest, along with the corresponding hyperparameters.

The next step is to repeat the same process for HistGradientBoosting
and compare the tuned models.

In [18]:
def objective_hist_gradient_boosting(trial):
    """
    Optuna objective function for HistGradientBoostingClassifier.

    Parameters
    ----------
    trial : optuna.trial.Trial
        Current optimization trial.

    Returns
    -------
    float
        Mean cross-validated F1-score.
    """
    model = HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 50),
        l2_regularization=trial.suggest_float(
            "l2_regularization", 1e-5, 10.0, log=True
        ),
        random_state=RANDOM_STATE,
    )

    model_pipeline = make_model_pipeline(preprocess_pipeline, model)

    cv_results = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=CV,
        scoring={"f1": "f1"},
        n_jobs=-1,
        return_train_score=False,
    )

    return np.mean(cv_results["test_f1"])

## HistGradientBoosting Objective Function

This function defines the search space for HistGradientBoosting
and evaluates each configuration using cross-validated F1-score.

Gradient boosting models are often strong candidates because they
capture non-linear relationships and interactions effectively.

In [19]:
study_hist_gradient_boosting = optuna.create_study(direction="maximize")
study_hist_gradient_boosting.optimize(
    objective_hist_gradient_boosting,
    n_trials=30,
)

[I 2026-03-20 17:35:14,285] A new study created in memory with name: no-name-80e0522d-2add-43a6-9566-cfb0e885b65d
[I 2026-03-20 17:35:14,732] Trial 0 finished with value: 0.7719511951195119 and parameters: {'learning_rate': 0.02398080371913462, 'max_iter': 327, 'max_depth': 10, 'min_samples_leaf': 32, 'l2_regularization': 0.05407823739681659}. Best is trial 0 with value: 0.7719511951195119.
[I 2026-03-20 17:35:15,004] Trial 1 finished with value: 0.7404989659239612 and parameters: {'learning_rate': 0.23193579368714892, 'max_iter': 158, 'max_depth': 12, 'min_samples_leaf': 24, 'l2_regularization': 0.19746599132508208}. Best is trial 0 with value: 0.7719511951195119.
[I 2026-03-20 17:35:15,308] Trial 2 finished with value: 0.7591973574237387 and parameters: {'learning_rate': 0.04908078423029397, 'max_iter': 308, 'max_depth': 5, 'min_samples_leaf': 37, 'l2_regularization': 0.00021470645254462127}. Best is trial 0 with value: 0.7719511951195119.
[I 2026-03-20 17:35:15,692] Trial 3 finished

In [20]:
print("Best HistGradientBoosting F1:", study_hist_gradient_boosting.best_value)
print("Best HistGradientBoosting params:")
study_hist_gradient_boosting.best_params

Best HistGradientBoosting F1: 0.7777113452585152
Best HistGradientBoosting params:


{'learning_rate': 0.06574632682519867,
 'max_iter': 113,
 'max_depth': 7,
 'min_samples_leaf': 18,
 'l2_regularization': 0.02253363898652837}

## HistGradientBoosting Tuning Result

This output shows the best cross-validated F1-score found by Optuna
for HistGradientBoosting, along with the corresponding hyperparameters.

At this point, both tuned candidate models can be compared directly
to determine which one should move to final evaluation.

In [21]:
tuned_results_df = pd.DataFrame(
    [
        {
            "model": "random_forest_tuned",
            "cv_f1_mean": study_random_forest.best_value,
        },
        {
            "model": "hist_gradient_boosting_tuned",
            "cv_f1_mean": study_hist_gradient_boosting.best_value,
        },
    ]
).sort_values(by="cv_f1_mean", ascending=False).reset_index(drop=True)

tuned_results_df

,model,cv_f1_mean
0,hist_gradient_boosting_tuned,0.777711
1,random_forest_tuned,0.774341


## Tuned Model Comparison

This table compares the best tuned versions of the candidate models.

The model with the highest cross-validated F1-score will be selected
for final training and evaluation on the test set.

In [22]:
best_tuned_model_name = tuned_results_df.iloc[0]["model"]
best_tuned_model_name

'hist_gradient_boosting_tuned'

## Final Model Candidate

This cell identifies the best tuned model according to cross-validated F1-score.

This model will now be trained on the full training set
and evaluated once on the untouched test set.

In [23]:
final_comparison_df = pd.concat(
    [
        all_results_df[["model", "cv_f1_mean"]],
        tuned_results_df.rename(columns={"cv_f1_mean": "cv_f1_mean"}),
    ],
    ignore_index=True,
).sort_values(by="cv_f1_mean", ascending=False).reset_index(drop=True)

final_comparison_df

,model,cv_f1_mean
0,hist_gradient_boosting_tuned,0.777711
1,random_forest_tuned,0.774341
2,logistic_regression,0.769401
3,random_forest,0.754871
4,hist_gradient_boosting,0.748608
5,decision_tree,0.709868
6,dummy,0.000000


## Final Model Comparison

This table compares all evaluated models:

- baseline models
- candidate models
- tuned models

The goal is to select the model with the best cross-validated F1-score,
regardless of its complexity.

It is possible that a simpler model outperforms more complex ones,
which would be preferable in terms of interpretability and robustness.

## Final Model Selection

The selected model is: `hist_gradient_boosting_tuned`

Justification:
- Highest cross-validated F1-score
- Stability across folds
- Simplicity vs performance trade-off

In [25]:
selected_model_name = final_comparison_df.iloc[0]["model"]
selected_model_name

'hist_gradient_boosting_tuned'

In [26]:
if selected_model_name == "dummy":
    final_estimator = DummyClassifier(strategy="most_frequent")

elif selected_model_name == "logistic_regression":
    final_estimator = LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "decision_tree":
    final_estimator = DecisionTreeClassifier(
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "random_forest":
    final_estimator = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

elif selected_model_name == "hist_gradient_boosting":
    final_estimator = HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
    )

elif selected_model_name == "random_forest_tuned":
    final_estimator = RandomForestClassifier(
        **study_random_forest.best_params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

elif selected_model_name == "hist_gradient_boosting_tuned":
    final_estimator = HistGradientBoostingClassifier(
        **study_hist_gradient_boosting.best_params,
        random_state=RANDOM_STATE,
    )

else:
    raise ValueError(f"Unknown selected model: {selected_model_name}")

In [27]:
final_model_pipeline = make_model_pipeline(
    preprocess_pipeline=preprocess_pipeline,
    model=final_estimator,
)

final_model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## Final Training

This cell trains the selected model using the full training set.

At this point:

- all model selection decisions have already been made,
- hyperparameter tuning has already been completed if necessary,
- and the model is now ready for a single final evaluation on the test set.

In [29]:
y_pred = final_model_pipeline.predict(X_test)

if hasattr(final_model_pipeline, "predict_proba"):
    y_proba = final_model_pipeline.predict_proba(X_test)[:, 1]
else:
    y_proba = None

In [30]:
test_results = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
}

if y_proba is not None:
    test_results["roc_auc"] = roc_auc_score(y_test, y_proba)

test_results_df = pd.DataFrame([test_results])
test_results_df

,accuracy,f1,roc_auc
0,0.821229,0.761194,0.829974


## Final Test Performance

This table reports the final performance of the selected model
on the untouched test set.

This is the first and only time the test set is used in the notebook,
which ensures a fair and unbiased evaluation.

In [31]:
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.842     0.873     0.857       110
           1      0.785     0.739     0.761        69

    accuracy                          0.821       179
   macro avg      0.813     0.806     0.809       179
weighted avg      0.820     0.821     0.820       179



In [32]:
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix

array([[96, 14],
       [18, 51]])

## Detailed Evaluation

The classification report and confusion matrix provide a more complete view
of model behavior than a single metric.

In particular, they help identify:

- whether the model favors one class,
- whether recall is weaker than precision,
- and which types of classification errors are more frequent.

# Final Conclusions

This notebook completed the first full modeling cycle of the Titanic project.

## Main outcomes

- A baseline performance level was established using simple models.
- Multiple classifiers were compared under the same cross-validation setup.
- The strongest candidate models were tuned with Optuna.
- A final model was selected based on cross-validated F1-score.
- The selected model was evaluated on the test set.
- Error analysis and interpretation were performed.

## Key methodological strengths

- The preprocessing pipeline was reused consistently.
- The test set remained untouched until final evaluation.
- Model selection was based on quantitative evidence.
- Simpler models were not discarded unnecessarily.

## Next steps

- refine feature engineering if needed,
- analyze subgroup failures more deeply,
- consider calibration or threshold adjustment,
- and prepare the model pipeline for deployment.